# Shreve Week 12 — Poisson Processes & Jump Diffusion

**Week 12** — Poisson Processes & Jump Diffusion

This notebook teaches **poisson processes & jump diffusion** in the style of our graduate probability notebook: definitions from **Shreve**, intuition and examples from **Baxter & Rennie**, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve | Baxter & Rennie |
|------|-------|--------|-----------------|
| 1 | **Poisson process** | Ch. 11.1 | Ch. 3.7 |
| 2 | **Compound Poisson** | Ch. 11.2 | Ch. 6 |
| 3 | **Jump-diffusion model** | Ch. 11.3 | Ch. 6.2 |
| 4 | **Merton jump model** | Ch. 11.3 | Shreve |
| — | **Problem set** | Ch. 11 | App. 3 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. **Shreve** (*Stochastic Calculus for Finance II*) — rigorous measure-theoretic treatment; see chapter pointers in each section.
5. **Baxter & Rennie** (*Financial Calculus*) — market intuition, replication, and worked examples; see spotlight sections.

**References:**
- **Shreve** Vol II — Ch. 11 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Baxter & Rennie**, *Financial Calculus* — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Poisson Process

$N_t$ with $N_0=0$, independent increments, $N_t - N_s \sim \text{Poisson}(\lambda(t-s))$.

Inter-arrival times are $\text{Exp}(\lambda)$.

**Shreve Ch. 11.1:** counting processes.


In [ ]:
def simulate_poisson(lam, T, seed=120):
    rng = np.random.default_rng(seed)
    t, N = 0.0, 0
    times = [0]
    counts = [0]
    while t < T:
        dt = rng.exponential(1/lam)
        t += dt
        if t < T:
            N += 1
            times.append(t)
            counts.append(N)
    times.append(T)
    counts.append(N)
    return np.array(times), np.array(counts)

lam, T = 3.0, 5.0
times, counts = simulate_poisson(lam, T)
plt.step(times, counts, where="post")
plt.title(f"Poisson process λ={lam}")
plt.show()
print(f"N_T mean = {counts[-1]}, theory λT = {lam*T}")


---
# Part 2 — Compound Poisson

$J_t = \sum_{i=1}^{N_t} Y_i$ with jump sizes $Y_i$ i.i.d.

Used for cumulative random jumps.

**Shreve Ch. 11.2:** compound Poisson process.


In [ ]:
rng = np.random.default_rng(121)
lam, T = 2.0, 3.0
n_jumps = rng.poisson(lam*T)
jump_sizes = rng.normal(0, 0.5, n_jumps)
J_T = jump_sizes.sum()
print(f"Compound Poisson J_T: mean sim over many runs...")
means = []
for s in range(5000):
    nj = rng.poisson(lam*T)
    means.append(rng.normal(0, 0.5, nj).sum())
print(f"E[J_T] = {np.mean(means):.4f} (theory 0)")


---
# Part 3 — Jump-Diffusion SDE

$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t + S_t\, dJ_t$

Combines continuous diffusion with jumps.

**Shreve Ch. 11.3:** jump-diffusion models.


In [ ]:
def simulate_merton_jump(S0, mu, sigma, lam, jump_mu, jump_sig, T, n, seed=122):
    rng = np.random.default_rng(seed)
    dt = T/n
    S = S0
    for _ in range(n):
        dW = rng.normal(0, np.sqrt(dt))
        # Poisson jump in interval
        nj = rng.poisson(lam*dt)
        jump = sum(rng.normal(jump_mu, jump_sig, nj)) if nj > 0 else 0
        S = S * np.exp((mu-0.5*sigma**2)*dt + sigma*dW + jump)
    return S

paths = [simulate_merton_jump(100, 0.08, 0.15, 2.0, -0.1, 0.2, 1, 252, seed=s)
         for s in range(20)]
for p in paths:
    plt.plot(p, alpha=0.7)
plt.title("Merton jump-diffusion paths")
plt.show()


---
# Part 4 — Merton Model & Fat Tails

Jumps create **heavier tails** than pure GBM — better fit to short-term options.

**Shreve Ch. 11.3:** Merton (1976) jump-diffusion.


In [ ]:
# Compare return distributions: GBM vs jump
rng = np.random.default_rng(123)
n = 50_000
gbm_rets = rng.normal(0.05/252, 0.2/np.sqrt(252), n)
jump_rets = []
for _ in range(n):
    r = rng.normal(0.05/252, 0.2/np.sqrt(252))
    if rng.random() < 2*1/252:
        r += rng.normal(-0.1, 0.2)
    jump_rets.append(r)
fig, ax = plt.subplots()
ax.hist(gbm_rets, bins=80, density=True, alpha=0.5, label="GBM")
ax.hist(jump_rets, bins=80, density=True, alpha=0.5, label="Jump")
ax.legend()
ax.set_title("Daily returns: GBM vs jump-diffusion")
plt.show()


**Your turn:** How does jump intensity $\lambda$ affect option implied volatility skew?


---
## Baxter & Rennie note — Beyond pure diffusion

Baxter & Rennie focus on **continuous** models (Ch. 3–6). Jump processes are covered rigorously in **Shreve Ch. 11**.

B&R **Ch. 6** ("Bigger models") discusses extensions: more assets, numeraires, and completeness — when continuous models suffice and when markets are incomplete.

**Read:** B&R Ch. 6 intro + Shreve Ch. 11.1 for Poisson processes.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Show inter-arrival times of Poisson($\lambda$) are Exp($\lambda$).

2. For compound Poisson with $E[Y]=0$, show $E[J_t]=0$.

3. Write log-return SDE for Merton model.

4. Why do jumps matter for short-dated options?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $P(T_1>t)=P(N_t=0)=e^{-\lambda t}$ → Exp($\lambda$).

**2.** $E[J_t]=E[N_t]E[Y]=\lambda t \cdot 0=0$.

**3.** $d\log S = (\mu-\sigma^2/2)dt + \sigma dW + d(\sum Y_i)$.

**4.** Short maturity: jump risk dominates diffusion; skew from asymmetric jumps.

</details>


---
## Further reading

### Shreve
- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 11 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

### Baxter & Rennie
- **Shreve** Ch. 11 — jump processes (primary for this week).
- **Baxter & Rennie**, Ch. 6 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf): model completeness, extensions; contrasts with jump incompleteness.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
